# **PHASE 04 — LABEL / TARGERT COLUMN CREATION**
### *Forward Return Threshold Labeling Strategy*
- Tranforming historical data into a supervised learning dataset for predicting profitable trades.

**How it works**
- “If I buy this stock today, will I make at least 5% profit within the next 10 days?”
- if ≥ 5% >>> 1 (BUY) else Not
- model learns which stocks today will gain ≥5% within 10 days

**How our Code Implements This**
- **Step 1** — Get Future Price >>> **LEAD(close, 10)** >>> This gets the closing price 10 records later >>>
- **Step 2** — Calculate Future Return >>> **future_return = (future_price / current_price ) - 1**
- **Step 3** — Apply Profit Threshold >>> PROFIT_THRESHOLD = 0.05 >>> if future_return = 10% -> 1

### Helpers

In [2]:
import os
from datetime import datetime
import numpy as np
import pandas as pd

# ================ CONFIG ===================

pd.set_option('display.max_columns', None)

# CSV Folder
CSV_FOLDER = "output_csv"
os.makedirs(CSV_FOLDER, exist_ok=True)


os.makedirs("log", exist_ok=True) # Create log folder if doesn't exit
# dynamically names a log file using the current date & time
LOG_FILE = os.path.join("log", f"loader_log_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt")

# method to add log msg into log file & prints them to console
def log(msg):
    with open(LOG_FILE, "a", encoding="utf-8") as f: # "a" mode → appends messages instead of overwriting
        f.write(msg + "\n")
    print(msg)

### Create DuckDB Connection

In [3]:
import os
import duckdb

DB_FOLDER = "database"
DB_PATH = os.path.join(DB_FOLDER, "cse_data.db")

con = duckdb.connect(database=DB_PATH)

In [15]:
# ================ Creating labels for 10-day return ==========================================================

# Create labeled table with:
#  - future closing price (10 days ahead)
#  - future return %
#  - classification label (buy or not)
#
# f""" allows Python to insert variables like {PROFIT_THRESHOLD} into SQL
#
# Using a CTE (temp) to compute future price once for efficiency
# instead of recalculating LEAD() multiple times
#
# LEAD(close, 10)
#     → gets closing price 10 rows ahead / after the current date (actualy not not days, rcords) if today 26.03.05 then look for 26.03.15
# PARTITION BY symbol
#     → calculation done separately for each stock
# ORDER BY date
#     → ensures rows are in time order
#
# Return formula:
#     future_return = (future_price / current_price) - 1
#
# Target label rule:
#     return ≥ threshold → 1 (profitable → buy signal)
#     return < threshold → 0 (not profitable)

log("=== Phase 3: Creating labels for 10-day return ===")


cols = ["target_buy_10d"]
for c in cols:
    con.execute(f"ALTER TABLE stocks_features_table DROP COLUMN IF EXISTS {c}")
con.execute("DROP TABLE IF EXISTS temp_future_price_table") 
con.execute("DROP TABLE IF EXISTS temp_returns_table") 
con.execute("DROP TABLE IF EXISTS stocks_features_labeled_10d_table") 


# Threshold for profitable trade, 0.05 - 5% profit in 10 days
PROFIT_THRESHOLD = 0.05

# Computing future closing price (10 records ahead/future),we cant here use past data bcs we wanna train model for future
# Right we dont have price record for lat 10 days if today 20/03 no data from 11-today(20) / 03 but that is ok we use that for training the model no 
# need to all at advanced we can use that trained model for today prediction although it is old 10 day, we train model from 1st day to today - 10days, 
# totaly fine
con.execute("""
    CREATE OR REPLACE TABLE temp_future_price_table AS
    SELECT 
        symbol,
        date, 
        close,
            -- looks n rows ahead (future)
            LEAD(close, 10) OVER (PARTITION BY symbol ORDER BY date) AS future_close_10d
    FROM stocks_features_clean_table
    WHERE close > 0
""")

# Compute future return
# After 10 days how the actual return has happeneded, is it a gain or loss relative to the today price if today price 100, after day days 
# it has become 110, then return is (110/100) - 1 = 1.1 - 1 = 0.1 is the gain
con.execute(f"""
    CREATE OR REPLACE TABLE temp_returns_table AS
    SELECT 
        symbol,
        date,
        future_close_10d,
        close,
           ((future_close_10d / close) - 1) AS raw_future_return,

        CASE 
           WHEN (future_close_10d / close - 1) > 1 THEN 1
           WHEN (future_close_10d / close - 1) < -1 THEN -1
           ELSE (future_close_10d / close - 1)
        END AS future_return_10d,

        -- if calculated future_return_10d value is > PROFIT_THRESHOLD value >>> 1 can buy
        COALESCE(
            CASE 
            WHEN (future_close_10d / close - 1) >= {PROFIT_THRESHOLD} THEN 1
            ELSE 0
            END, 0
        ) AS target_buy_10d
       
    FROM temp_future_price_table
    WHERE future_close_10d IS NOT NULL
""")

# copy stocks_features_clean_table to new table named stocks_features_labeled_10d_table to add target features
con.execute(f"""
    CREATE OR REPLACE TABLE stocks_features_labeled_10d_table AS
    SELECT *,
    FROM stocks_features_clean_table;
""")

# Crate support columns
con.execute("ALTER TABLE stocks_features_labeled_10d_table ADD COLUMN IF NOT EXISTS future_close_10d DOUBLE")
con.execute("ALTER TABLE stocks_features_labeled_10d_table ADD COLUMN IF NOT EXISTS future_return_10d DOUBLE")

# Create label columns
con.execute("ALTER TABLE stocks_features_labeled_10d_table ADD COLUMN IF NOT EXISTS target_buy_10d INTEGER")

# target_buy_10d - target/y column, buy or not 1 or 0
con.execute("""
    UPDATE stocks_features_labeled_10d_table AS s
    SET 
        -- COALESCE(value1, value2, ..., valueN) returns the first non-NULL value from the list
        target_buy_10d = COALESCE(v.target_buy_10d, 0),
        future_close_10d = v.future_close_10d,
        future_return_10d = v.future_return_10d
    FROM temp_returns_table AS v
    WHERE s.symbol = v.symbol AND s.date = v.date;
""")

con.execute("""
    UPDATE stocks_features_labeled_10d_table
    SET target_buy_10d = 0
    WHERE target_buy_10d IS NULL;
""")

log("Label creation complete : 10-day later closed price col, return % col & target_buy_10d binary col added")


# ================= EXPORT LABELED DATASET =====================================================================

CSV_FOLDER = "output_csv"
os.makedirs(CSV_FOLDER, exist_ok=True)
LABELED_CSV_PATH = os.path.join(CSV_FOLDER, "features_labeled_10d.csv")

log(f"Exporting labeled dataset to {LABELED_CSV_PATH} ...")

# HEADER → includes column names
# DELIMITER ',' → comma-separated format
con.execute(f"""
    COPY stocks_features_labeled_10d_table
    TO '{LABELED_CSV_PATH}'
    (HEADER, DELIMITER ',');
""")

log(f"Labeled dataset exported dataset ready for ML training → {LABELED_CSV_PATH}")

# table > temp_future_price   temp_returns
# col   > future_return_10d

=== Phase 3: Creating labels for 10-day return ===
Label creation complete : 10-day later closed price col, return % col & target_buy_10d binary col added
Exporting labeled dataset to output_csv\features_labeled_10d.csv ...
Labeled dataset exported dataset ready for ML training → output_csv\features_labeled_10d.csv


#  QUICK PREVIEW

### temp_returns_table

In [16]:
temp_future_price_df = con.execute("SELECT * FROM temp_returns_table").fetchdf()
temp_future_price_df.sample(10)

,symbol,date,future_close_10d,close,raw_future_return,future_return_10d,target_buy_10d
8605,LPL.N0000,2026-01-16,13.60,14.10,-0.035461,-0.035461,0
450,REXP.N0000,2026-02-19,384.00,409.00,-0.061125,-0.061125,0
2298,SHOT.N0000,2026-01-26,20.40,20.50,-0.004878,-0.004878,0
5604,MGT.N0000,2026-01-02,38.50,40.60,-0.051724,-0.051724,0
2627,MDL.N0000,2026-01-12,23.80,20.80,0.144231,0.144231,1
906,LDEV.N0000,2026-02-03,21.00,22.00,-0.045455,-0.045455,0
1253,AMSL.N0000,2026-02-16,14.00,15.30,-0.084967,-0.084967,0
3800,LOLC.N0000,2026-02-24,551.25,570.25,-0.033319,-0.033319,0
1890,CINS.X0000,2026-01-29,1750.50,1720.00,0.017733,0.017733,0
248,HUNT.N0000,2026-01-12,1498.50,1349.25,0.110617,0.110617,1


### stocks_features_labeled_10d_table

In [17]:
stocks_features_labeled_df = con.execute("SELECT * FROM stocks_features_labeled_10d_table").fetchdf()
stocks_features_labeled_df.sample(10)

,company,symbol,volume,trades,prev_close,open,high,low,close,change,change_percentage,date,dup_index,close_1d,close_3d,close_5d,close_10d,close_20d,close_60d,high_20,low_20,std_vol_20,ret_1,ret_3,ret_5,ret_10,ret_20,ret_60,std_close_5,std_close_10,std_close_20,range_5,atr_14,TR,volume_ratio,volume_z,avg_vol_10,avg_vol_20,liquidity_score,ma_ratio_5,ma_ratio_10,ma_ratio_20,price_position,breakout_flag,momentum_score,volatility_score,turnover_ratio,trend_angle,range_position,liquidity_rank,future_close_10d,future_return_10d,target_buy_10d
9543,ASIRI SURGICAL HOSPITAL PLC,AMSL.N0000,1526004,434,14.00,14.30,17.5,13.50,17.10,3.10,22.14,2026-03-11,1,14.00,14.10,14.0,15.0,14.00,NaN,17.10,14.00,4.525988e+05,0.221429,0.212766,0.221429,0.140000,0.221429,NaN,1.375863,0.953881,0.828188,4.0,0.900000,4.00,6.851269,2.879528,193327.0,222733.05,19.728419,1.168033,1.170431,1.141522,0.921053,0,0.205143,1.139733,0.000284,-0.064511,1.000000,92.0,NaN,NaN,0
1487,ACCESS ENGINEERING PLC,AEL.N0000,1561128,369,72.30,73.00,75.0,72.20,74.80,2.50,3.46,2026-02-17,1,72.30,71.40,71.6,72.0,75.60,NaN,74.80,71.40,3.946156e+05,0.034578,0.047619,0.044693,0.038889,-0.010582,NaN,1.298075,0.994708,1.146666,5.0,1.871429,2.80,2.246181,2.194829,635437.1,695014.35,4.929983,1.030303,1.036729,1.026486,0.774194,0,0.038475,1.176783,0.000236,-0.104962,1.000000,57.0,74.1,-0.009358,0
3198,HATTON NATIONAL BANK PLC,HNB.X0000,498323,539,392.25,391.00,404.0,391.00,399.25,7.00,1.78,2026-02-16,1,392.25,368.75,369.0,375.5,343.25,NaN,399.25,349.25,1.297158e+05,0.017846,0.082712,0.081978,0.063249,0.163146,NaN,14.598801,11.328461,11.926005,35.5,8.375000,13.00,2.927600,2.529433,135802.0,170215.55,7.405168,1.050658,1.064880,1.074769,1.023904,1,0.046166,13.083140,0.001082,1.248496,1.000000,123.0,368.5,-0.077019,0
10470,SOFTLOGIC FINANCE PLC,CRL.N0000,1252499,148,5.90,5.90,6.2,5.80,6.10,0.20,3.39,2026-02-17,1,5.90,5.90,5.8,6.3,5.40,NaN,6.30,5.30,1.209198e+06,0.033898,0.033898,0.051724,-0.031746,0.129630,NaN,0.130384,0.202485,0.316228,0.6,0.492857,0.40,1.933745,0.500160,677793.1,647706.50,0.967182,1.013289,1.035654,1.070175,0.647059,0,0.026117,0.189183,0.000118,0.040752,0.800000,61.0,6.1,0.000000,0
6906,LOLC GENERAL INSURANCE PLC,LGIL.N0000,1365377,144,7.30,7.20,7.4,6.50,6.50,-0.80,-10.96,2026-03-19,1,7.90,8.10,8.2,8.3,8.50,NaN,8.80,6.50,2.967450e+05,-0.177215,-0.197531,-0.207317,-0.216867,-0.235294,NaN,0.687023,0.556277,0.491908,2.0,0.492857,1.40,10.928157,4.180141,208764.1,124941.20,45.681232,0.841969,0.817610,0.790274,-0.916667,0,-0.194176,0.608776,0.000105,-0.060226,0.000000,125.0,NaN,NaN,0
7134,WATAWALA PLANTATIONS PLC,WATA.N0000,147810,178,46.00,46.50,46.5,45.90,46.00,0.00,0.00,2026-02-16,1,46.00,47.90,47.9,49.0,49.00,NaN,49.30,46.00,7.019583e+04,0.000000,-0.039666,-0.039666,-0.061224,-0.061224,NaN,1.043072,0.913175,0.960414,3.2,0.985714,0.60,1.151758,0.277450,100697.5,128334.20,0.319555,0.975817,0.967403,0.954654,0.050000,0,-0.024145,0.987572,0.001204,-0.111203,0.000000,138.0,42.6,-0.073913,0
11573,SOFTLOGIC HOLDINGS PLC,SHL.N0000,14718019,1608,7.40,7.70,9.2,7.70,8.80,1.40,18.92,2026-02-27,1,7.40,7.30,7.3,7.0,6.40,NaN,8.80,6.50,3.739826e+06,0.189189,0.205479,0.205479,0.257143,0.375000,NaN,0.709225,0.514782,0.585325,2.2,0.678571,1.80,5.662457,3.240469,2296362.6,2599228.60,18.349013,1.164021,1.181208,1.206306,1.000000,0,0.207667,0.626112,0.000109,0.026992,1.000000,22.0,NaN,NaN,0
4991,AMANA BANK PLC,ABL.N0000,186666,162,30.00,30.00,30.1,29.30,29.50,-0.50,-1.67,2026-02-02,1,30.00,29.80,29.6,30.2,32.40,NaN,32.00,29.10,4.273073e+05,-0.016667,-0.010067,-0.003378,-0.023179,-0.089506,NaN,0.204939,0.184089,0.630351,0.8,0.728571,0.80,0.478147,-0.476773,421165.4,390394.40,0.227968,0.989269,0.988275,0.981534,0.288462,0,-0.013983,0.283766,0.000868,-0.031955,0.137931,81.0,30.0,0.016949,0
9932,HOTEL SIGIRIYA PLC,HSIG.N0000,701,3,95.00,95.10,98.0,95.10,95.90,0.90,0.95,2026-02-09,1,95.00,94.60,93.4,96.5,100.00,NaN,101.50,93.40,4.378440e+03,0.009474,0.013742,0.026767,-0.006218,-0.041000,NaN,0.906642,1.003

In [18]:
print("no of columns : ", len(stocks_features_labeled_df.columns),"\n\n",stocks_features_labeled_df.columns)

no of columns :  53 

 Index(['company', 'symbol', 'volume', 'trades', 'prev_close', 'open', 'high',
       'low', 'close', 'change', 'change_percentage', 'date', 'dup_index',
       'close_1d', 'close_3d', 'close_5d', 'close_10d', 'close_20d',
       'close_60d', 'high_20', 'low_20', 'std_vol_20', 'ret_1', 'ret_3',
       'ret_5', 'ret_10', 'ret_20', 'ret_60', 'std_close_5', 'std_close_10',
       'std_close_20', 'range_5', 'atr_14', 'TR', 'volume_ratio', 'volume_z',
       'avg_vol_10', 'avg_vol_20', 'liquidity_score', 'ma_ratio_5',
       'ma_ratio_10', 'ma_ratio_20', 'price_position', 'breakout_flag',
       'momentum_score', 'volatility_score', 'turnover_ratio', 'trend_angle',
       'range_position', 'liquidity_rank', 'future_close_10d',
       'future_return_10d', 'target_buy_10d'],
      dtype='object')


### Class Imbalance Testing

In [19]:
import duckdb

# Load db
con = duckdb.connect("database/cse_data.db")

counts = con.execute("""
    SELECT 
        target_buy_10d,
        COUNT(*) AS count,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS percentage
    FROM stocks_features_labeled_10d_table
    GROUP BY target_buy_10d
    ORDER BY target_buy_10d
""").fetch_df()

print(counts)

   target_buy_10d  count  percentage
0               0  10296       87.36
1               1   1490       12.64


### Close DuckDB Connection

In [21]:
if 'con' in globals():  # Check if connection exists
    try:
        con.close()        # Close it
        print("DuckDB connection closed.")
    except Exception as e:
        print("Error closing DuckDB:", e)
    finally:
        del con             # Delete the variable from memory

# stocks_features_labeled_10d_table